# Task 2: Cloud VM Inference — All 4 Models

Side-by-side comparison of Florence-2-large, Qwen2.5-VL-3B, Qwen2.5-VL-7B, and Phi-4-multimodal.

On A100 (80 GB): all 4 models loaded simultaneously.
On T4 (16 GB): load one at a time, clear VRAM between models.

In [ ]:
import torch, time
from PIL import Image
import matplotlib.pyplot as plt

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
A100 = torch.cuda.get_device_properties(0).total_memory > 40e9
print(f'Mode: {"All models simultaneously" if A100 else "One at a time (swap)"}')

## Load Test Image

In [ ]:
# Change this to your image
import requests
from io import BytesIO
url = 'https://images.unsplash.com/photo-1583511655857-d19b40a7a54e?w=640'
image = Image.open(BytesIO(requests.get(url, headers={'User-Agent':'Mozilla/5.0'}).content)).convert('RGB')
plt.figure(figsize=(6,6)); plt.imshow(image); plt.axis('off'); plt.title(f'{image.width}x{image.height}'); plt.show()

## 1. Florence-2-large

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor

f2_proc = AutoProcessor.from_pretrained('microsoft/Florence-2-large-ft', trust_remote_code=True)
f2_model = AutoModelForCausalLM.from_pretrained('microsoft/Florence-2-large-ft', torch_dtype=torch.float16, trust_remote_code=True).to('cuda')
print('Florence-2-large loaded.')

In [ ]:
for task in ['<OD>', '<CAPTION>', '<DENSE_REGION_CAPTION>', '<OCR>']:
    inputs = f2_proc(text=task, images=image, return_tensors='pt').to('cuda', torch.float16)
    t = time.time()
    ids = f2_model.generate(**inputs, max_new_tokens=1024, do_sample=False, num_beams=3)
    elapsed = time.time() - t
    text = f2_proc.batch_decode(ids, skip_special_tokens=False)[0]
    parsed = f2_proc.post_process_generation(text, task=task, image_size=(image.width, image.height))
    print(f'\n{task} ({elapsed:.2f}s):')
    print(f'  {parsed}')

In [ ]:
if not A100:
    del f2_model, f2_proc; torch.cuda.empty_cache(); print('Cleared VRAM')

## 2. Qwen2.5-VL-3B

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor as QwenProcessor

q3_proc = QwenProcessor.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct')
q3_model = Qwen2_5_VLForConditionalGeneration.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct', torch_dtype=torch.bfloat16, device_map='auto')
print('Qwen2.5-VL-3B loaded.')

In [ ]:
prompts = [
    'Describe what you see in this image.',
    'How many distinct objects are in this image? List them.',
    'What is the main subject of this image?',
]
for p in prompts:
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':p}]}]
    text = q3_proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = q3_proc(text=[text], images=[image], return_tensors='pt', padding=True).to(q3_model.device)
    t = time.time()
    ids = q3_model.generate(**inputs, max_new_tokens=256)
    elapsed = time.time() - t
    out = q3_proc.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    print(f'\nPrompt: {p} ({elapsed:.2f}s)')
    print(f'  {out[:300]}')

In [ ]:
if not A100:
    del q3_model, q3_proc; torch.cuda.empty_cache(); print('Cleared VRAM')

## 3. Qwen2.5-VL-7B

In [ ]:
q7_proc = QwenProcessor.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct')
q7_model = Qwen2_5_VLForConditionalGeneration.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct', torch_dtype=torch.bfloat16, device_map='auto')
print('Qwen2.5-VL-7B loaded.')

In [ ]:
for p in prompts:
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':p}]}]
    text = q7_proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = q7_proc(text=[text], images=[image], return_tensors='pt', padding=True).to(q7_model.device)
    t = time.time()
    ids = q7_model.generate(**inputs, max_new_tokens=256)
    elapsed = time.time() - t
    out = q7_proc.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    print(f'\nPrompt: {p} ({elapsed:.2f}s)')
    print(f'  {out[:300]}')

In [ ]:
if not A100:
    del q7_model, q7_proc; torch.cuda.empty_cache(); print('Cleared VRAM')

## 4. Phi-4-multimodal

In [ ]:
phi_proc = AutoProcessor.from_pretrained('microsoft/Phi-4-multimodal-instruct', trust_remote_code=True)
phi_model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-4-multimodal-instruct', torch_dtype=torch.bfloat16, trust_remote_code=True, device_map='auto', _attn_implementation='eager')
print('Phi-4-multimodal loaded.')

In [ ]:
for p in prompts:
    full = f'<|user|>\n<|image_1|>\n{p}<|end|>\n<|assistant|>\n'
    inputs = phi_proc(full, images=[image], return_tensors='pt').to(phi_model.device)
    t = time.time()
    ids = phi_model.generate(**inputs, max_new_tokens=256)
    elapsed = time.time() - t
    out = phi_proc.batch_decode(ids, skip_special_tokens=True)[0]
    if '<|assistant|>' in out: out = out.split('<|assistant|>')[-1].strip()
    print(f'\nPrompt: {p} ({elapsed:.2f}s)')
    print(f'  {out[:300]}')

## VRAM Usage Summary

In [ ]:
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'VRAM reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB')
print(f'VRAM total:     {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')